# 6. Stage Logging (Expanded)

Tracks and prints transitions between motion stages.

---

```python
def log_stage(self, stage, description):
    if self.current_stage != stage:
        self.current_stage = stage
        print(f"\n[Time {self.time_:.2f}s] Stage {stage}: {description}")
```

---

## 🧠 Big Picture: Why This Exists

This function is not about motion—it is about **observability**.

It allows you to answer:

```text
"What is the robot doing right now?"
```

Without this, your system becomes a **black box**:

* You send commands
* The robot moves
* But you don’t know *which part of the program is active*

---

## 🔄 The Core Idea: Detecting Transitions

### Key line:

```python
if self.current_stage != stage:
```

---

### What this means:

```text
Only act when the stage CHANGES
```

---

### Why is this important?

Your control loop runs at **50 Hz**:

```text
50 times per second → ControlLoop()
```

Without this check:

```python
print("Stage 2: Raising arm")
```

would execute:

```text
50 times per second × 3 seconds = 150 prints
```

---

### Result without protection:

```text
Stage 2
Stage 2
Stage 2
Stage 2
...
```

→ Completely unusable output

---

### With this function:

```text
Stage 2: Raising arm   ← printed ONCE
```

---

## 🧱 State Variable: `self.current_stage`

```python
self.current_stage = -1
```

---

### What it represents:

```text
The last stage that was printed
```

---

### Flow:

| Step            | Value     |
| --------------- | --------- |
| Start           | -1        |
| Enter Stage 1   | becomes 1 |
| Stay in Stage 1 | remains 1 |
| Enter Stage 2   | becomes 2 |

---

### This creates a **finite-state memory**

---

## 🕒 Time-Stamped Logging

```python
print(f"\n[Time {self.time_:.2f}s] Stage {stage}: {description}")
```

---

### Output example:

```text
[Time 3.00s] Stage 2: Raising left arm
```

---

### Why include time?

This allows you to:

* Verify timing behavior
* Debug stage transitions
* Correlate motion with code

---

### 🧠 Engineering Insight

This is critical in robotics because:

> Motion is time-dependent, not just state-dependent

---

## 🔍 Debugging Value (VERY IMPORTANT)

This function is your **primary debugging tool** for:

### 1. Timing errors

If you see:

```text
Stage 2 at 2.5s instead of 3.0s
```

→ Your timing logic is wrong

---

### 2. Stuck states

If output stops at:

```text
Stage 2
```

→ Your program never reaches Stage 3

---

### 3. Fast transitions

If you see:

```text
Stage 1
Stage 2
Stage 3
```

immediately:

→ Your time variable is broken

---

## ⚙️ Relationship to ControlLoop

Inside your `ControlLoop`, you have:

```python
if t < d:
    self.log_stage(1, "Stabilizing")
elif t < 2*d:
    self.log_stage(2, "Raising arm")
```

---

### This creates a **time-based state machine**

```text
Time → determines stage → triggers log
```

---

## 🧠 Software Engineering Pattern

This function implements a classic pattern:

> **Edge-triggered event logging**

---

### Compare:

| Type            | Behavior              |
| --------------- | --------------------- |
| Level-triggered | logs continuously     |
| Edge-triggered  | logs only on change ✅ |

---

## 🤖 Robotics Insight

In real robot systems:

* Logging is often the **only visibility**
* You cannot always:

  * See internal variables
  * Pause execution
  * Step through code

So:

> Good logging = controllability + debuggability

---

## 🔄 Analogy

Think of this like:

```text
Elevator display:
"Floor 3"
```

It only updates when:

* You reach a new floor

Not continuously while staying on the same floor.

---

## 🤖 RL Perspective

In reinforcement learning, this is analogous to:

```text
Episode phase transitions
```

Examples:

* Start of episode
* Goal reached
* Failure state

---

## 🔥 Subtle but Important Detail

```python
self.current_stage = stage
```

This line must come **before printing completes**, otherwise:

* Multiple threads or rapid calls could cause duplicate logs

---

## 🚀 Summary

This function:

| Role                | Description             |
| ------------------- | ----------------------- |
| Detects transitions | Stage changes only      |
| Prevents spam       | Avoids repeated prints  |
| Adds timestamps     | Enables timing analysis |
| Aids debugging      | Reveals system behavior |

---

## 🧠 Teaching Insight

This is a great place to emphasize:

> “In robotics, knowing *what the system is doing* is as important as controlling it.”

Students should learn:

* Logging is not optional
* Logging is a **core control tool**

